## Generate from trained model

- The model checkpoint used: trained in MacBook Pro (M4 Pro) Chips for approx. 1hr. 
- Batch size = 32, steps = 4200, context length = 256. 
- Therefore the total tokens processed in this training is `34,406,400`
- The loss went down to approx 1.49 at the end of this training (considering the vocab size is only 10k)

In [ ]:
# infrence from trained model
from run_train_model import generate
import torch
import torch.nn.functional as F
from model import Transformer as Model
from tokenizer_optimized import Tokenizer

model_ckpt = "train_logs/run_v0_1/ckpt_iter_4199.pt"
vocab_file = "outputs/vocab_tinystories_optimized.json"
merge_file = "outputs/merges_tinystories_optimized.json"

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
print(f"using device: {device}")

# init Tokenizer
tokenizer = Tokenizer.from_files(vocab_file, merge_file, special_tokens=["<|endoftext|>"])
eos_id = tokenizer.special_token_to_id.get("<|endoftext|>")

# reconstruct model
model_args = dict(
    vocab_size=10000, 
    context_length=256, 
    n_head=16, 
    num_layers=4, 
    d_model=512, 
    d_ff=1344,
    theta=10000.0
)

print(f"Loading model from {model_ckpt}...")
model = Model(**model_args).to(device)

# load weights
checkpoint = torch.load(model_ckpt, map_location=device)
state_dict = checkpoint['model'] # it has keys: "model", "optimizer", "iteration".
model.load_state_dict(state_dict)
model.eval()

# 开始调戏模型
prompts = [
    "Once upon a time, there was a little boy named Tim.",
    "The cat sat on the mat and",
    "Lily found a magic wand in the garden, and she"
]

print("-" * 30)
for p in prompts:
    print(f"Prompt: {p}")
    full_output, _ = generate(model, tokenizer=tokenizer, context=p, max_new_tokens=256, temperature=0.8, top_p=0.9, eos_id=tokenizer.special_token_to_id.get("<|endoftext|>"), context_length=256, device=device)
    print(f"Generated: {full_output}")
    print("=" * 80)

using device: mps
Loading model from /Users/siyuanfang/Desktop/toLearn/cs336/assignment1-basics/train_logs/run_20260126_132030/ckpt_iter_4199.pt...
------------------------------
Prompt: Once upon a time, there was a little boy named Tim.
Generated: Once upon a time, there was a little boy named Tim. Tim was a very intelligent boy. He loved to run and play all day. One day, Tim's mom told him to go to the store. Tim was very excited and could not wait to go outside.
When Tim got to the store, he saw a big box. He wanted to open it, but it was too heavy. Tim was sad and didn't know what to do. He went to his mom and asked for help. His mom said, "Let's ask dad to help you. Maybe you can help me open the box."
When they got to the store, Tim and his mom got a big box of cookies. They were happy and opened the box together. Tim learned that he should always ask for help when he needs it. And they all lived happily ever after.
<|endoftext|>
Prompt: The cat sat on the mat and
Generated: The